# Data Walkthrough — Bronze to Serving

This single notebook walks through **every** table in the platform, in pipeline
order: Bronze (lake) → Silver (lake) → Gold dims/facts/identity → Summary → Serving.

It reads from the local DuckDB file `transformation/dbt_project/local_retail.duckdb`,
the single catalog for the local simulation:

- Bronze/Silver tables are loaded from Flink Parquet by
  `scripts/local/load_iceberg_to_duckdb.py` (the `-DbtSource iceberg` path).
- Gold/Summary/Serving tables are built by `dbt run --target local`.

**Prerequisite** — run the full local stack first so the DuckDB file is populated:

```powershell
.\scripts\local\run_local_stack.ps1 -Task all
```

Then run the cells top-to-bottom. Each layer prints row counts, columns, samples,
and a couple of aggregations so you can see the data flow end-to-end in one place.

Reference diagrams: `docs/data-model/erd.md`, `docs/data-model/dimensional-model.md`,
`docs/data-model/platform-layers.md`.

## Setup

One helper: `q(sql)` runs a read-only query against `local_retail.duckdb` and
prints rows as a list. `df(sql)` returns a pandas DataFrame for richer display.
Both use the repo `.venv`'s `duckdb` (no host pyarrow/pandas needed for Bronze/Silver
since they are already loaded into DuckDB by `load_iceberg_to_duckdb.py`).

In [ ]:
from pathlib import Path

import duckdb

DB_PATH = Path.cwd().resolve().parent / "transformation" / "dbt_project" / "local_retail.duckdb"
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"{DB_PATH} not found. Run `scripts/local/run_local_stack.ps1 -Task all` first."
    )

con = duckdb.connect(str(DB_PATH), read_only=True)


def q(sql):
    try:
        rows = con.execute(sql).fetchall()
        cols = [d[0] for d in con.description]
    except duckdb.Error as e:
        print(f"[not built yet] {e}")
        return []
    print(" | ".join(cols))
    print("-" * 80)
    for r in rows:
        print(" | ".join(str(x) for x in r))
    return rows


def df(sql):
    return con.execute(sql).df()


print(f"Connected to {DB_PATH.name} (read-only)")

## 1. Bronze (lake raw capture)

Three bronze tables — one per source. POS lands as Parquet directly (no Kafka);
clickstream + inventory land via Kafka → Flink `*_bronze_job` → Iceberg.

In [ ]:
q("""
select 'clickstream_events' as tbl, count(*) as rows from bronze.clickstream_events
union all select 'inventory_events', count(*) from bronze.inventory_events
union all select 'pos_transactions', count(*) from bronze.pos_transactions
order by tbl
""")

In [ ]:
for t in ("clickstream_events", "inventory_events", "pos_transactions"):
    print(f"=== bronze.{t} ===")
    try:
        cols = [r[0] for r in con.execute(f"describe bronze.{t}").fetchall()]
        print("columns:", ", ".join(cols))
        print(con.execute(f"select * from bronze.{t} limit 3").df().to_string(index=False))
    except duckdb.Error as e:
        print(f"[not built yet] {e}")
    print()

In [ ]:
print("Clickstream by event_type:")
q("""select event_type, count(*) from bronze.clickstream_events
     group by 1 order by 2 desc""")

print("\nClickstream by platform:")
q("""select platform, count(*) from bronze.clickstream_events
     group by 1 order by 2 desc""")

## 2. Silver (lake curated)

One silver table: `silver.inventory_hourly` — Flink `inventory_silver_job` rolls
up bronze inventory deltas into hourly buckets (kappa path, ADR-007). Columns
are **hourly deltas** (`qty_delta_hour`) and gross receipts (`qty_received_hour`),
NOT running balances — the running on-hand balance is computed in
`finance.fact_inventory_snapshot`.

In [ ]:
q("""select count(*) as rows from silver.inventory_hourly""")
print()
q("""select snapshot_date_key, snapshot_hour,
           sum(qty_delta_hour) as delta, sum(qty_received_hour) as received
    from silver.inventory_hourly
    group by 1, 2 order by 1, 2 limit 10""")

## 3. Gold — Conformed dimensions

Kimball dims built by dbt. `dim_product` is SCD Type 2 (the only SCD2 dim); the
rest are Type 1. `dim_customer` is enriched with consent + RFM attributes.

In [ ]:
q("""
select 'dim_date' as tbl, count(*) as rows from finance.dim_date
union all select 'dim_store', count(*) from finance.dim_store
union all select 'dim_product', count(*) from finance.dim_product
union all select 'dim_customer', count(*) from marketing.dim_customer
order by tbl
""")

In [ ]:
print("dim_product SCD2 — current vs historical versions:")
q("""select is_current, count(*) as versions
    from finance.dim_product group by 1 order by 1""")
print()
print("Sample SCD2 history (one product with >1 version):")
q("""select product_id, effective_from, effective_to, is_current, brand, unit_cost
    from finance.dim_product
    where product_id in (
        select product_id from finance.dim_product
        group by product_id having count(*) > 1 limit 1
    )
    order by product_id, effective_from""")

In [ ]:
print("dim_customer by loyalty_tier:")
q("""select loyalty_tier, count(*) from marketing.dim_customer
    group by 1 order by 2 desc""")
print("\ndim_customer by rfm_segment:")
q("""select rfm_segment, count(*) from marketing.dim_customer
    group by 1 order by 2 desc""")
print("\nConsent coverage:")
q("""select
       sum(case when marketing_consent then 1 else 0 end) as marketing_consent,
       sum(case when analytics_consent then 1 else 0 end) as analytics_consent,
       count(*) as total
    from marketing.dim_customer""")

## 4. Gold — Fact tables

Three fact tables, one per source grain:

- `fact_sales` — line-item grain `(transaction_id, line_item_number)`
- `fact_inventory_snapshot` — hourly grain `(snapshot_date_key, snapshot_hour, product_key, store_key)`
- `fact_customer_session` — session grain `(session_id)`

In [ ]:
q("""
select 'fact_sales' as tbl, count(*) as rows from finance.fact_sales
union all select 'fact_inventory_snapshot', count(*) from finance.fact_inventory_snapshot
union all select 'fact_customer_session', count(*) from marketing.fact_customer_session
order by tbl
""")

In [ ]:
print("fact_sales — net revenue + units by date:")
q("""select date_key, sum(quantity_sold) as units, sum(net_revenue) as net_rev
    from finance.fact_sales
    where not is_voided
    group by 1 order by 1 limit 10""")
print("\nfact_sales — voided rate:")
q("""select sum(case when is_voided then 1 else 0 end) as voided,
       count(*) as total,
       round(100.0 * sum(case when is_voided then 1 else 0 end) / count(*), 2) as pct_voided
    from finance.fact_sales""")

In [ ]:
print("fact_inventory_snapshot — on-hand range:")
q("""select min(quantity_on_hand) as min_qoh, max(quantity_on_hand) as max_qoh,
           avg(quantity_on_hand)::int as avg_qoh
    from finance.fact_inventory_snapshot""")
print("\nfact_inventory_snapshot — stockout hours (quantity_available = 0):")
q("""select count(*) as stockout_rows
    from finance.fact_inventory_snapshot where quantity_available = 0""")

In [ ]:
print("fact_customer_session — by platform:")
q("""select platform, count(*) as sessions,
       sum(case when converted then 1 else 0 end) as converted,
       round(100.0 * sum(case when converted then 1 else 0 end) / count(*), 2) as conv_pct
    from marketing.fact_customer_session
    group by 1 order by 2 desc""")
print("\nfact_customer_session — identified vs anonymous:")
q("""select
       sum(case when customer_key is not null then 1 else 0 end) as identified,
       sum(case when customer_key is null then 1 else 0 end) as anonymous,
       count(*) as total
    from marketing.fact_customer_session""")

## 5. Gold — Identity graph

`marketing.identity_graph` maps raw identifiers (loyalty_id, customer_id,
client_id) to a resolved `customer_key`. Public devices (clients shared across
10+ distinct customers) are excluded from the graph but kept in
`int_identity_resolution` for audit (method=`public_device_excluded`).

In [ ]:
print("identity_graph by identifier_type:")
q("""select identifier_type, count(*) from marketing.identity_graph
    group by 1 order by 2 desc""")
print("\nidentity_graph by resolution_method:")
q("""select resolution_method, count(*) from marketing.identity_graph
    group by 1 order by 2 desc""")
print("\nidentity_graph — distinct resolved customers:")
q("""select count(distinct customer_key) as distinct_customers
    from marketing.identity_graph""")

## 6. Summary (daily rollups)

One daily rollup per Gold fact — `marts.summary` schema. Each is a single-fact
daily aggregation (no fact-to-fact joins; drill-across only).

In [ ]:
q("""
select 'sales_daily_store' as tbl, count(*) as rows from summary.sales_daily_store
union all select 'inventory_daily_product_store', count(*) from summary.inventory_daily_product_store
union all select 'sessions_daily_platform', count(*) from summary.sessions_daily_platform
order by tbl
""")
print()
print("sales_daily_store — top 5 days by net_revenue:")
q("""select date_key, store_key, net_revenue, units_sold
    from summary.sales_daily_store order by net_revenue desc limit 5""")
print("\nsessions_daily_platform — conversion by platform:")
q("""select session_date_key, platform, session_count, converted_sessions,
       conversion_rate
    from summary.sessions_daily_platform order by session_date_key, platform limit 10""")

## 7. Serving (BI / dashboard contract)

`serving.customer_360_serving` is the consent-gated Customer 360 view at
`(customer_key, session_id)` grain — the dbt-managed BI/dashboard contract
(filtered to `marketing_consent = true`).

In [ ]:
q("""select count(*) as rows, count(distinct customer_key) as consented_customers
    from serving.customer_360_serving""")
print()
print("customer_360_serving — by loyalty_tier:")
q("""select loyalty_tier, count(*) as rows
    from serving.customer_360_serving group by 1 order by 2 desc""")
print("\nSample:")
q("""select customer_key, loyalty_id, loyalty_tier, rfm_segment,
       session_id, converted, platform
    from serving.customer_360_serving limit 5""")

## Notes

- This notebook is **read-only** — it never mutates the lake or warehouse.
- For paste-able non-notebook versions of individual queries, see
  `docs/runbooks/local-data-queries.md` (Bronze/Silver via `docker compose exec`)
  and `docs/data-model/identity-resolution.md` (identity graph).
- For **cloud** post-deploy smoke checks (EMR state, S3 bronze prefixes,
  Redshift row counts, dashboard HTTP), use
  `scripts/cloud/deploy_platform.ps1 -Env <dev|prod> -Action verify` — that
  replaces the old `06_cloud_platform_verification.ipynb` notebook.
- If a section returns 0 rows, the corresponding upstream step has not run yet
  (e.g. Flink bronze job, `load_iceberg_to_duckdb.py`, or `dbt run`). Re-run
  `scripts/local/run_local_stack.ps1 -Task all`.